In [ ]:
from pathlib import Path

print("Thư mục notebook đang chạy:")
print(Path.cwd())

print("\nCác file hiện có trong data_processed:")
folder = Path("data_processed")

if folder.exists():
    for file in folder.iterdir():
        print("-", file.name)
else:
    print("Chưa tìm thấy thư mục data_processed")

: 

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

# 1. Khai báo thư mục dữ liệu
DATA_DIR = Path("data_processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

sales_path = DATA_DIR / "sales_train_evaluation.csv"
calendar_path = DATA_DIR / "calendar.csv"
prices_path = DATA_DIR / "sell_prices.csv"

# 2. Kiểm tra các file đầu vào
for file_path in [sales_path, calendar_path, prices_path]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file: {file_path.resolve()}"
        )

print("Đang đọc dữ liệu M5...")

# 3. Đọc dữ liệu bán hàng
sales = pd.read_csv(sales_path)
calendar = pd.read_csv(calendar_path)
prices = pd.read_csv(prices_path)

# Chỉ chọn cửa hàng CA_1 và nhóm hàng thực phẩm FOODS_
sales = sales[
    (sales["store_id"] == "CA_1")
    & (sales["dept_id"].str.startswith("FOODS_", na=False))
].copy()

if sales.empty:
    raise ValueError(
        "Không tìm thấy sản phẩm thuộc CA_1 và nhóm FOODS_."
    )

print("Số sản phẩm được chọn:", len(sales))

# 4. Chuyển dữ liệu bán hàng từ dạng ngang sang dạng dọc
day_columns = [
    col for col in sales.columns
    if col.startswith("d_")
]

sales_long = sales.melt(
    id_vars=[
        "item_id", "dept_id", "cat_id",
        "store_id", "state_id"
    ],
    value_vars=day_columns,
    var_name="d",
    value_name="sales"
)

sales_long["sales"] = pd.to_numeric(
    sales_long["sales"], errors="coerce"
).fillna(0)

# 5. Ghép ngày và thông tin sự kiện
calendar["date"] = pd.to_datetime(calendar["date"])

data = sales_long.merge(
    calendar,
    on="d",
    how="left"
)

# 6. Ghép giá bán
data = data.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

data["sell_price"] = data["sell_price"].fillna(0)

# 7. Tạo các đặc trưng thời gian
data["month"] = data["date"].dt.month
data["year"] = data["date"].dt.year
data["day_of_month"] = data["date"].dt.day
data["day_of_week"] = data["date"].dt.dayofweek
data["week_of_year"] = (
    data["date"].dt.isocalendar().week.astype(int)
)
data["is_weekend"] = (
    data["day_of_week"] >= 5
).astype(int)

# 8. Tạo đặc trưng sự kiện
event_cols = ["event_name_1", "event_type_1",
              "event_name_2", "event_type_2"]

for col in event_cols:
    data[col] = data[col].fillna("Unknown").astype(str)

data["is_event"] = (
    (data["event_name_1"] != "Unknown")
    | (data["event_name_2"] != "Unknown")
).astype(int)

# 9. Tạo đặc trưng lịch sử bán hàng theo từng sản phẩm
data = data.sort_values(
    ["item_id", "date"]
).reset_index(drop=True)

grouped_sales = data.groupby("item_id")["sales"]

data["sales_lag_1"] = grouped_sales.shift(1)
data["sales_lag_7"] = grouped_sales.shift(7)
data["sales_lag_14"] = grouped_sales.shift(14)
data["sales_lag_28"] = grouped_sales.shift(28)

# Trung bình bán hàng chỉ dùng dữ liệu của các ngày trước đó
data["sales_roll_mean_7"] = data.groupby(
    "item_id"
)["sales"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).mean()
)

data["sales_roll_mean_28"] = data.groupby(
    "item_id"
)["sales"].transform(
    lambda s: s.shift(1).rolling(28, min_periods=28).mean()
)

# 10. Mã hóa các cột phân loại
categorical_cols = [
    "item_id", "dept_id", "cat_id", "store_id",
    "state_id", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2"
]

# Giữ lại mã sản phẩm gốc để cell sau dùng được item_id
data["item_id_original"] = data["item_id"]

for col in categorical_cols:
    if col == "item_id":
        continue

    data[col] = (
        data[col]
        .fillna("Unknown")
        .astype("category")
        .cat.codes
    )

# Tạo mã số riêng cho từng sản phẩm
item_codes = {
    item: code
    for code, item in enumerate(
        data["item_id_original"].drop_duplicates()
    )
}
data["item_id_code"] = data["item_id_original"].map(item_codes)

# 11. Đổi tên các cột phân loại theo đúng tên model yêu cầu
data = data.rename(columns={
    "dept_id": "dept_id_code",
    "cat_id": "cat_id_code",
    "store_id": "store_id_code",
    "state_id": "state_id_code",
    "event_name_1": "event_name_1_code",
    "event_type_1": "event_type_1_code",
    "event_name_2": "event_name_2_code",
    "event_type_2": "event_type_2_code"
})

# 12. Khai báo các đặc trưng dùng cho Random Forest
features = [
    "item_id_code", "dept_id_code", "cat_id_code",
    "store_id_code", "state_id_code",
    "event_name_1_code", "event_type_1_code",
    "event_name_2_code", "event_type_2_code",
    "sell_price", "month", "year", "day_of_month",
    "day_of_week", "week_of_year", "is_weekend",
    "is_event", "sales_lag_1", "sales_lag_7",
    "sales_lag_14", "sales_lag_28",
    "sales_roll_mean_7", "sales_roll_mean_28"
]

required_cols = [
    "item_id_original", "date", "sales"
] + features

# Bỏ những dòng chưa đủ dữ liệu lịch sử
data = data.dropna(
    subset=[
        "sales_lag_1", "sales_lag_7",
        "sales_lag_14", "sales_lag_28",
        "sales_roll_mean_7", "sales_roll_mean_28"
    ]
).copy()

# Kiểm tra các cột trước khi lưu
missing_cols = [
    col for col in required_cols
    if col not in data.columns
]

if missing_cols:
    raise ValueError(
        f"Thiếu các cột cần thiết: {missing_cols}"
    )

data = data[required_cols].copy()

# Giữ nguyên item_id dạng chuỗi để cell dự báo ghép dữ liệu được
data = data.rename(
    columns={"item_id_original": "item_id"}
)

# 13. Chia tập train và test theo thời gian
data["date"] = pd.to_datetime(data["date"])

train_end = pd.Timestamp("2016-03-27")
test_end = pd.Timestamp("2016-04-24")

train_df = data[data["date"] <= train_end].copy()
test_df = data[
    (data["date"] > train_end)
    & (data["date"] <= test_end)
].copy()

# Kiểm tra dữ liệu
if train_df.empty or test_df.empty:
    raise ValueError(
        "Train hoặc test đang rỗng. "
        "Hãy kiểm tra ngày trong calendar.csv."
    )

# 14. Lưu dữ liệu để cell huấn luyện đọc
train_df.to_csv(DATA_DIR / "train.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)

print("\nĐÃ TIỀN XỬ LÝ XONG")
print("Số dòng train:", len(train_df))
print("Số dòng test:", len(test_df))
print("Số sản phẩm train:", train_df["item_id"].nunique())
print("Ngày cuối train:", train_df["date"].max().date())
print("Ngày đầu test:", test_df["date"].min().date())
print("Ngày cuối test:", test_df["date"].max().date())
print("Đã lưu:", DATA_DIR / "train.csv")
print("Đã lưu:", DATA_DIR / "test.csv")

display(train_df.head())
display(test_df.head())


In [ ]:

# CELL 3: ĐỀ XUẤT LƯỢNG HÀNG CẦN BỔ SUNG

from pathlib import Path
import numpy as np
import pandas as pd

# 1. Khai báo thư mục dữ liệu
DATA_DIR = Path("data_processed")

# 2. Kiểm tra dữ liệu đầu vào
required_files = [
    "train.csv",
    "rf_test_predictions.csv",
    "sigma_e_by_item.csv",
]

missing_files = [
    name for name in required_files
    if not (DATA_DIR / name).exists()
]

if missing_files:
    print("CHƯA THỂ TÍNH LƯỢNG HÀNG CẦN NHẬP.")
    print("Thư mục dữ liệu:", DATA_DIR.resolve())
    print("Các file còn thiếu:")

    for name in missing_files:
        print("-", name)

    raise FileNotFoundError(
        "Hãy chạy thành công cell huấn luyện Random Forest "
        "để tạo rf_test_predictions.csv và sigma_e_by_item.csv, "
        "sau đó chạy lại Cell 3."
    )

# 3. Đọc dữ liệu lịch sử và kết quả dự báo
train_df = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["date"],
)

test_predictions = pd.read_csv(
    DATA_DIR / "rf_test_predictions.csv",
    parse_dates=["date"],
)

sigma_e_by_item = pd.read_csv(
    DATA_DIR / "sigma_e_by_item.csv",
)

# 4. Kiểm tra các cột bắt buộc
if not {"item_id", "date", "sales"}.issubset(train_df.columns):
    raise ValueError(
        "train.csv phải có đủ các cột item_id, date và sales."
    )

if not {"item_id", "date", "y_pred"}.issubset(
    test_predictions.columns
):
    raise ValueError(
        "rf_test_predictions.csv phải có item_id, date và y_pred."
    )

if not {"item_id", "sigma_e"}.issubset(
    sigma_e_by_item.columns
):
    raise ValueError(
        "sigma_e_by_item.csv phải có item_id và sigma_e."
    )

# 5. Thiết lập các tham số mô phỏng
LEAD_TIME_DAYS = 6
REVIEW_INTERVAL_DAYS = 1
STOCK_COVER_DAYS = 3
Z = 1.65

# Tổng thời gian cần đáp ứng nhu cầu
T = LEAD_TIME_DAYS + REVIEW_INTERVAL_DAYS

# Ngày lập kế hoạch là ngày cuối cùng trong tập train
as_of_date = train_df["date"].max()

print("Ngày lập kế hoạch:", as_of_date.date())
print("Thời gian chờ hàng:", LEAD_TIME_DAYS, "ngày")
print("Chu kỳ xem xét:", REVIEW_INTERVAL_DAYS, "ngày")
print("Khoảng dự báo:", T, "ngày")

# 6. Tính doanh số trung bình mỗi ngày trong 28 ngày gần nhất
history_start = as_of_date - pd.Timedelta(days=27)

history_28_days = train_df[
    train_df["date"].between(history_start, as_of_date)
].copy()

average_sales = (
    history_28_days
    .groupby("item_id", as_index=False)["sales"]
    .mean()
    .rename(columns={"sales": "avg_daily_sales"})
)

if average_sales.empty:
    raise ValueError(
        "Không có dữ liệu bán hàng trong 28 ngày gần nhất."
    )

# 7. Mô phỏng lượng tồn kho hiện tại
inventory = average_sales[
    ["item_id", "avg_daily_sales"]
].copy()

# Giả định tồn kho bằng mức bán trung bình nhân số ngày bao phủ
inventory["on_hand"] = np.ceil(
    inventory["avg_daily_sales"] * STOCK_COVER_DAYS
).astype(int)

# Giả định ban đầu không có đơn hàng đang về
inventory["on_order"] = 0

inventory["lead_time_days"] = LEAD_TIME_DAYS

# 8. Lấy dự báo nhu cầu trong T ngày tiếp theo
period_end = as_of_date + pd.Timedelta(days=T)

forecast_period = test_predictions[
    (test_predictions["date"] > as_of_date)
    & (test_predictions["date"] <= period_end)
].copy()

if forecast_period.empty:
    raise ValueError(
        f"Không có dữ liệu dự báo trong khoảng "
        f"{as_of_date.date()} đến {period_end.date()}. "
        "Hãy kiểm tra lại file rf_test_predictions.csv."
    )

# Tổng hợp nhu cầu dự báo của từng sản phẩm
demand_by_item = (
    forecast_period
    .groupby("item_id", as_index=False)
    .agg(
        forecast_demand=("y_pred", "sum"),
        forecast_days=("date", "nunique"),
    )
)

# Bảo đảm có đủ T ngày dự báo cho từng sản phẩm
incomplete = demand_by_item[
    demand_by_item["forecast_days"] != T
]

if not incomplete.empty:
    raise ValueError(
        f"Có {len(incomplete)} sản phẩm chưa đủ "
        f"{T} ngày dự báo. Hãy kiểm tra dữ liệu test."
    )

# 9. Kết hợp tồn kho, nhu cầu dự báo và sai số mô hình
result = (
    inventory
    .merge(demand_by_item, on="item_id", how="inner")
    .merge(sigma_e_by_item, on="item_id", how="left")
)

if result.empty:
    raise ValueError(
        "Không ghép được dữ liệu tồn kho với dữ liệu dự báo. "
        "Hãy kiểm tra mã item_id giữa các file."
    )

if result["sigma_e"].isna().any():
    missing_count = int(result["sigma_e"].isna().sum())
    raise ValueError(
        f"Có {missing_count} sản phẩm chưa có sigma_e."
    )

if (result["sigma_e"] < 0).any():
    raise ValueError("sigma_e không được âm.")

# 10. Tính tồn kho an toàn
# S = Z * sigma_e * căn bậc hai của T
result["safety_stock"] = (
    Z * result["sigma_e"] * np.sqrt(T)
)

# 11. Tính vị thế tồn kho
# I = tồn kho hiện có + hàng đang về
result["inventory_position"] = (
    result["on_hand"] + result["on_order"]
)

# 12. Tính lượng hàng đề xuất bổ sung
# Q = max(0, nhu cầu dự báo + tồn kho an toàn - vị thế tồn kho)
result["suggested_qty"] = np.ceil(
    (
        result["forecast_demand"]
        + result["safety_stock"]
        - result["inventory_position"]
    ).clip(lower=0)
).astype(int)

# 13. Đưa ra khuyến nghị
result["recommendation"] = np.where(
    result["suggested_qty"] > 0,
    "Cần nhập",
    "Chưa cần nhập",
)

# 14. Tạo bảng kết quả
recommendations = result[
    [
        "item_id",
        "forecast_demand",
        "sigma_e",
        "safety_stock",
        "on_hand",
        "on_order",
        "inventory_position",
        "suggested_qty",
        "recommendation",
    ]
].sort_values(
    "suggested_qty",
    ascending=False,
).reset_index(drop=True)

# 15. Hiển thị và lưu kết quả
print("\nTOP 20 SẢN PHẨM CẦN BỔ SUNG NHIỀU NHẤT:")
display(recommendations.head(20))

output_path = DATA_DIR / "replenishment_recommendations.csv"

recommendations.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nĐã hoàn thành tính toán.")
print("Tổng số sản phẩm:", len(recommendations))
print(
    "Số sản phẩm cần nhập:",
    int((recommendations["suggested_qty"] > 0).sum()),
)
print("File kết quả:", output_path.resolve())


In [ ]:

from pathlib import Path

print("Thư mục dữ liệu:", DATA_DIR.resolve())

for name in [
    "train.csv",
    "test.csv",
    "rf_test_predictions.csv",
    "sigma_e_by_item.csv",
]:
    path = DATA_DIR / name
    print(f"{name}: {'ĐÃ CÓ' if path.exists() else 'CHƯA CÓ'}")
